# Stage 2. Control image quality

## 1. Setup

### 1.1 Imports

In [ ]:
import logging

from acid.config import load_config, prepare_output
from acid.image_quality_control.display_qc import plot_quality
from acid.image_quality_control.fov_quality import flag_quality, measure_quality
from acid.utils.metadata.filtering import filter_metadata_by_splits
from acid.utils.metadata.loading import load_metadata
from acid.utils.metadata.saving import save_metadata_dataframe


### 1.2 Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### 1.3 Load configuration

In [ ]:
CONFIG_FILE = None  # "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
prepare_output(config)
stage_cfg = config.quality_control
paths_cfg = config.shared.paths

## 2. Input Data Preparation

In [ ]:
metadata_df, metadata_file_name = load_metadata(stage_cfg.metadata)
metadata_df = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=stage_cfg.dataset_split
).copy()
if metadata_df.empty:
    raise ValueError("No training images selected")

## 3. Quality control
### 3.1 Helpers

### 3.1 Measure and flag training images

In [ ]:
metadata_df, num_channels, mean_over_std_column_names, skewness_column_names = (
    measure_quality(metadata_df, stage_cfg)
)
flagged_metadata_df = flag_quality(metadata_df, mean_over_std_column_names, stage_cfg)

## 4. Save metadata

In [ ]:
metadata_path = save_metadata_dataframe(
    flagged_metadata_df, stage_cfg.metadata, config.project_identity.project_name
)

## 5. Inspect results

In [ ]:
flagged_metadata_df.head()

### 5.1 QC plots

In [ ]:
plot_quality(
    flagged_metadata_df,
    mean_over_std_column_names,
    skewness_column_names,
    num_channels,
    stage_cfg,
    config.project_identity.project_name,
)

### 5.2 Inspect values near a target

In [ ]:
display_cfg = stage_cfg.visualization
for selection in (display_cfg.mean_over_std, display_cfg.skewness):
    column = f"{selection.metric}{stage_cfg.metadata.dataframe_columns.channel_measurement_separator}{selection.channel}"
    values = metadata_df[column]
    selected = metadata_df.loc[
        values.between(
            selection.value - abs(selection.range),
            selection.value + abs(selection.range),
            inclusive="neither",
        ),
        [stage_cfg.metadata.dataframe_columns.fov_column_name, column],
    ]
    display(selected)